<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/benchmark/developer_device_platform/litert_cli_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# LiteRT / LiteRT-LM Benchmarking Using Developer Device Platform (DDP)


This notebook demonstrates how to benchmark edge models on real devices using [Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) (DDP), Google Cloud's managed device lab.

You can benchmark [LiteRT](https://github.com/google-ai-edge/litert) (`.tflite`) and [LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) (`.litertlm`) models on a DDP device with a single `litert benchmark --ddp` command from the [LiteRT CLI](https://github.com/google-ai-edge/LiteRT-CLI).


## 🛠️ 1. Environment Setup & Installation

In [ ]:
# @title Setup your GCP project for DDP
# @markdown Set your GCP project id ([Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) is in Preview and its sessions are billed to this project, so it needs billing enabled; see the [quickstart](https://docs.cloud.google.com/developer-device-platform/quickstart)). The cell logs in to gcloud and enables the Device Run API.

ddp_gcp_project = "your-own-gcp-project-id" # @param {type:"string"}

if not ddp_gcp_project or ddp_gcp_project == "your-own-gcp-project-id":
  raise ValueError("Error: Please specify a valid GCP Project ID in the form above.")

# Login to gcloud and set Application Default Credentials via Colab Auth
from google.colab import auth
auth.authenticate_user()

# Enable the devicerun API
!gcloud services enable devicerun.googleapis.com --project {ddp_gcp_project}


## 🛠️ 2. Use LiteRT CLI to benchmark LiteRT / LiteRT-LM models

In [ ]:
# @title Install `LiteRT CLI`
# @markdown Installs the nightly build of the [LiteRT CLI](https://github.com/google-ai-edge/LiteRT-CLI), which ships the `--ddp` benchmark target.

# Install LiteRT CLI
!pip install litert-cli-nightly


In [ ]:
# @title Run `litert benchmark` to benchmark LiteRT models on DDP

# @markdown Set device type you want to benchmark. Device ids come from the DDP catalog, which you can
# @markdown list with `gcloud beta device-run devices list` (see the
# @markdown [DDP quickstart](https://docs.cloud.google.com/developer-device-platform/quickstart)).
ddp_device = "pa3q-35" # @param {type:"string"}

# Download and benchmark litert models
# Download EfficientNet-B1 model directly from HuggingFace (No conversion needed)
!litert download litert-community/efficientnet_b1 --file efficientnet_b1.tflite --output efficientnet

# Benchmark EfficientNet-B1 on a DDP device with CPU, then GPU
!litert benchmark efficientnet/efficientnet_b1.tflite --ddp --device {ddp_device} --cpu --gcp-project {ddp_gcp_project}
!litert benchmark efficientnet/efficientnet_b1.tflite --ddp --device {ddp_device} --gpu --gcp-project {ddp_gcp_project}

## 🛠️ 3. Use LiteRT CLI to benchmark LiteRT-LM models

In [ ]:
# @title Run `litert benchmark` to benchmark LiteRT-LM models on DDP

# @markdown Set device type you want to benchmark. Device ids come from the DDP catalog, which you can
# @markdown list with `gcloud beta device-run devices list` (see the
# @markdown [DDP quickstart](https://docs.cloud.google.com/developer-device-platform/quickstart)).
ddp_device = "m2q-36" # @param {type:"string"}

# Download and benchmark litert models

# Download Qwen3 0.6B int4 model directly from HuggingFace (No conversion needed)
!litert download litert-community/Qwen3-0.6B --file Qwen3-0.6B_dynamic_wi4b32_afp32.litertlm --output qwen3

# Benchmark a LiteRT-LM bundle (prefill and decode tokens/s) on the same device's GPU
!litert benchmark qwen3/Qwen3-0.6B_dynamic_wi4b32_afp32.litertlm --ddp --device {ddp_device} --gpu --gcp-project {ddp_gcp_project}


## 🛠️ 4. Visualize the results

In [ ]:
# @title Define `show_results` to read the result files `litert benchmark --ddp` saved
# @markdown `litert benchmark --ddp` saves each run's result files and device log under `~/.cache/litert-cli/ddp/<session>/<accelerator>-<device>/` and prints `Output files saved to:` with that path.
# @markdown Run this cell once; the two cells below call `show_results`. It only reads those files and does not call DDP.
import re
import statistics
from pathlib import Path

import pandas as pd

# --- Metrics schema -----------------------------------------------------------
# benchmark_model writes results.pb (tflite.tools.benchmark.BenchmarkResult) for
# a .tflite model; litert_lm_advanced_main writes metrics.pb
# (litert.lm.proto.LitertLmMetricsList) for a .litertlm bundle. Both are binary
# protobufs. No PyPI package ships their schemas, so fetch the public .proto
# files from LiteRT and LiteRT-LM, compile them to a descriptor set, and build
# the message classes at runtime. A descriptor set, rather than generated
# _pb2.py modules, keeps this independent of whichever protobuf version the
# runtime has installed. protobuf and grpcio are pinned to the preinstalled
# versions so pip picks a matching grpcio-tools instead of upgrading them.
import grpc, google.protobuf
!pip install -q grpcio-tools "protobuf=={google.protobuf.__version__}" "grpcio=={grpc.__version__}"
PROTO_DIR = "/content/litert_proto"
LITERT_RAW = "https://raw.githubusercontent.com/google-ai-edge/litert/main"
LITERT_LM_RAW = "https://raw.githubusercontent.com/google-ai-edge/LiteRT-LM/main"
BENCHMARK_PROTO = "tflite/tools/benchmark/proto/benchmark_result.proto"
!mkdir -p {PROTO_DIR}/tflite/tools/benchmark/proto {PROTO_DIR}/runtime/proto
!curl -sfL {LITERT_RAW}/{BENCHMARK_PROTO} -o {PROTO_DIR}/{BENCHMARK_PROTO}
for _p in ["engine.proto", "litert_lm_metrics.proto"]:
  !curl -sfL {LITERT_LM_RAW}/runtime/proto/{_p} -o {PROTO_DIR}/runtime/proto/{_p}
!python -m grpc_tools.protoc -I{PROTO_DIR} --include_imports \
    --descriptor_set_out={PROTO_DIR}/metrics.desc \
    {PROTO_DIR}/{BENCHMARK_PROTO} \
    {PROTO_DIR}/runtime/proto/engine.proto \
    {PROTO_DIR}/runtime/proto/litert_lm_metrics.proto

from google.protobuf import descriptor_pb2, descriptor_pool, message_factory
_pool = descriptor_pool.DescriptorPool()
for _f in descriptor_pb2.FileDescriptorSet.FromString(
    Path(f"{PROTO_DIR}/metrics.desc").read_bytes()).file:
  _pool.Add(_f)
BenchmarkResult = message_factory.GetMessageClass(
    _pool.FindMessageTypeByName("tflite.tools.benchmark.BenchmarkResult"))
MetricsList = message_factory.GetMessageClass(
    _pool.FindMessageTypeByName("litert.lm.proto.LitertLmMetricsList"))

# --- Reading results ----------------------------------------------------------
# The CLI saves each job's files here, and every reader below looks here.
RESULTS_DIR = Path.home() / ".cache/litert-cli/ddp"

TFLITE_METRICS = ["init_ms", "inference_avg_ms", "peak_ram_mb"]
LM_METRICS = ["prefill_tok_s", "decode_tok_s", "ttft_s", "init_ms", "peak_ram_mb"]
# `litert benchmark` leaves the first iteration(s) of a .litertlm run out of the
# medians it prints (--warmup-runs, default 1); so do these helpers.
LM_WARMUP_ITERATIONS = 1


def _median(values):
  values = [v for v in values if v]  # unset or zero means not measured
  return statistics.median(values) if values else None


def read_tflite_metrics(job_dir) -> dict:
  """Returns init, average inference time and peak RAM from results.pb.

  benchmark_model's measured process writes the file; the first process, which
  writes the model caches and runs no inference, writes warmup_results.pb.
  `runs` is how many inferences the average is over. A failed run writes no
  file, so a missing file yields empty values.
  """
  pb_path = Path(job_dir) / "results.pb"
  if not pb_path.exists():
    return {**dict.fromkeys(TFLITE_METRICS), "runs": 0}
  result = BenchmarkResult.FromString(pb_path.read_bytes())
  return {
      "init_ms": result.latency_metrics.init_ms,
      "inference_avg_ms": result.latency_metrics.avg_ms,
      # unset or zero means not measured
      "peak_ram_mb": result.memory_metrics.peak_mem_mb or None,
      "runs": result.misc_metrics.num_runs,
  }


def read_lm_metrics(job_dir) -> dict:
  """Returns prefill/decode tokens/s, TTFT, init and peak RAM from metrics.pb.

  The measured process writes one entry per iteration; the warm-up process
  before it writes no file. The values cover the iterations after the first
  LM_WARMUP_ITERATIONS (all of them when there are no more): medians of the
  speeds and TTFT, the highest peak RAM, and the init the process logged.
  `iters` is how many iterations that is. A failed run writes no file, so a
  missing file yields empty values.
  """
  pb_path = Path(job_dir) / "metrics.pb"
  if not pb_path.exists():
    return {**dict.fromkeys(LM_METRICS), "iters": 0}
  runs = MetricsList.FromString(pb_path.read_bytes()).metrics
  if not runs:
    return {**dict.fromkeys(LM_METRICS), "iters": 0}
  if len(runs) > LM_WARMUP_ITERATIONS:
    runs = runs[LM_WARMUP_ITERATIONS:]
  init_us = runs[0].init_phase_durations_us.get("Init Total")
  return {
      "prefill_tok_s": _median(m.prefill_turns[-1].tokens_per_second
                               for m in runs if m.prefill_turns),
      "decode_tok_s": _median(m.decode_turns[-1].tokens_per_second
                              for m in runs if m.decode_turns),
      "ttft_s": _median(m.time_to_first_token_seconds for m in runs),
      "init_ms": init_us / 1000 if init_us else None,
      # The peak over the measured iterations, as the CLI prints it.
      "peak_ram_mb": max((m.peak_mem_mb for m in runs if m.peak_mem_mb),
                         default=None),
      "iters": len(runs),
  }


def gpu_delegated(job_dir) -> bool | None:
  """True if the OpenCL delegate loaded in the measured process.

  The result files do not record this, so it reads logcat.txt. The job runs
  the binary more than once; the measured process logged the last benchmark
  line, and only its lines count. None when there is no logcat.
  """
  logcat = Path(job_dir) / "logcat.txt"
  if not logcat.exists():
    return None
  lines = logcat.read_text(errors="replace").splitlines()
  # A logcat line starts "MM-DD HH:MM:SS.mmm PID TID".
  pids = [re.match(r"\d\d-\d\d \d\d:\d\d:\d\d\.\d+\s+(\d+)\s", line)
          for line in lines]
  benchmark = [pid for line, pid in zip(lines, pids)
               if "BenchmarkInfo:" in line
               or "[benchmark_litert_model.h" in line]
  if benchmark and benchmark[-1]:
    measured = benchmark[-1].group(1)
    lines = [line for line, pid in zip(lines, pids)
             if pid and pid.group(1) == measured]
  return any("LITERT_CL" in line for line in lines)


def exit_code(job_dir) -> str | None:
  """The exit code the job's run record ends with; None without a record.

  On the device, the CLI's run script appends "<process> process: exit N" for
  each process it runs to benchmark_run.txt (.tflite) or provenance.txt
  (.litertlm). The last one is the job's exit code.
  """
  for name in ["benchmark_run.txt", "provenance.txt"]:
    record = Path(job_dir) / name
    if record.exists():
      exits = re.findall(r"process: exit (\d+)",
                         record.read_text(errors="replace"))
      return exits[-1] if exits else None
  return None


def newest_jobs(kind) -> dict[tuple[str, str], Path]:
  """The newest job folder of each (device, backend) that ran a `kind` model.

  kind is "tflite" (benchmark_model) or "lm" (a .litertlm bundle). The CLI
  names each job folder <backend>-<device>; the newest is the one modified
  last. A folder with no files (their download failed) or no result files is
  skipped with a note.
  """
  newest = {}
  for job in sorted(RESULTS_DIR.glob("session-*/*-*")):
    if not job.is_dir():
      continue
    files = {f.name for f in job.iterdir()}
    if not files:
      print(f"Skipping {job}: no files")
      continue
    if files & {"results.pb", "warmup_results.pb", "benchmark_run.txt"}:
      job_kind = "tflite"
    elif files & {"metrics.pb", "provenance.txt"}:
      job_kind = "lm"
    else:  # No result files: tell the two apart by the log.
      log = ((job / "logcat.txt").read_text(errors="replace")
             if "logcat.txt" in files else "")
      if "benchmark_litert_model" in log:
        job_kind = "tflite"
      elif "BenchmarkInfo" in log or "litert_lm_lib" in log:
        job_kind = "lm"
      else:
        print(f"Skipping {job}: no result files")
        continue
    if job_kind != kind:
      continue
    backend, _, device = job.name.partition("-")
    key = (device, backend)
    if key not in newest or job.stat().st_mtime > newest[key].stat().st_mtime:
      newest[key] = job
  return newest


def show_results(kind, read, metric_keys) -> pd.DataFrame | None:
  """Tabulates the newest run of each device and backend, then plots it.

  `read` turns a job folder into the metric columns (read_tflite_metrics or
  read_lm_metrics). Prints the table and the rows whose run failed or whose GPU
  run did not load the GPU delegate, then one bar chart per metric when there
  is more than one row. Returns None when no run was found.
  """
  rows = []
  for (device, backend), job in newest_jobs(kind).items():
    rows.append({
        "device": device,
        "backend": backend,
        "exit": exit_code(job),
        **read(job),
        "gpu_ok": gpu_delegated(job) if backend == "gpu" else None,
        "session": job.parent.name,
    })

  if not rows:
    print(f"No results found under {RESULTS_DIR} (the CLI prints 'Output files "
          "saved to: ...' after each run it saved) -- did the benchmark cell "
          "pass?")
    return None

  df = pd.DataFrame(rows).set_index(["device", "backend"]).sort_index()
  # A metric no run measured is all None; as floats it prints and plots as NaN.
  df[metric_keys] = df[metric_keys].astype(float)
  print(df.to_string(float_format=lambda v: f"{v:,.2f}"))

  # Alert on failures or missing GPU delegation.
  count = "runs" if "runs" in df else "iters"
  bad = df[(df.exit.notna() & (df.exit != "0")) | (df[count] == 0)
           | df.gpu_ok.eq(False)]
  if not bad.empty:
    print("\nFAILED or no GPU delegation: "
          + ", ".join(f"{d}/{b}" for d, b in bad.index))

  # One run has nothing to compare with: the table is the whole result.
  if len(df) > 1:
    for metric in metric_keys:
      df[metric].unstack("device").plot.bar(title=metric, figsize=(8, 3), rot=0)

  return df

In [ ]:
# @title Show the LiteRT results
# @markdown Table of the newest `.tflite` run per device and backend, and a bar chart per metric when there is more than one run.

tflite_df = show_results("tflite", read_tflite_metrics, TFLITE_METRICS)

In [ ]:
# @title Show the LiteRT-LM results
# @markdown Table of the newest `.litertlm` run per device and backend, and a bar chart per metric when there is more than one run.

lm_df = show_results("lm", read_lm_metrics, LM_METRICS)